### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [29]:
#%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [30]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [31]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [32]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [33]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [34]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [35]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [36]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [37]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [38]:
#tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [39]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [40]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [41]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [42]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [43]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [44]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [45]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [46]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [47]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [48]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [49]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [50]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [51]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Punto 1


In [52]:
np.random.seed(17)
idx = np.random.choice(len(fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes')).data), size=5, replace=False)

for doc_idx in idx:
    print("=" * 80)
    print(f"DOCUMENTO {doc_idx} — Clase: {newsgroups_train.target_names[y_train[doc_idx]]}")
    print("=" * 80)
    print(newsgroups_train.data[doc_idx][:500].strip())
    print("-" * 80)

    cossim = cosine_similarity(X_train[doc_idx], X_train)[0]

    top5 = np.argsort(cossim)[::-1][1:6]

    print("\n5 documentos más similares:")
    for rank, sim_idx in enumerate(top5, 1):
        sim_class = newsgroups_train.target_names[y_train[sim_idx]]
        print(f"  Top {rank} | Doc {sim_idx} | Similaridad: {cossim[sim_idx]:.4f} | Clase: {sim_class}")

    top1_idx = top5[0]
    print(f"\n--- TOP 1 (Doc {top1_idx}) — Clase: {newsgroups_train.target_names[y_train[top1_idx]]} ---")
    print(newsgroups_train.data[top1_idx][:500].strip())
    print("\n")

DOCUMENTO 10568 — Clase: talk.religion.misc
I recently read an article in a local paper written by an Islamic
  person who was upset with the way Islam has been portrayed by western media.
  When a terrorist action takes place in the middle east, it is always played
  up as an Islamic Terrorist.  However, when the a Serbian terrorist attacks
  the Croations, its not a Christian terrorist, its just a terrorist.
	I have often tried to explain this to some close friends who believe
  the press, that Islam is somehow tied to violence.  Often
--------------------------------------------------------------------------------

5 documentos más similares:
  Top 1 | Doc 6177 | Similaridad: 0.2893 | Clase: talk.religion.misc
  Top 2 | Doc 6331 | Similaridad: 0.2660 | Clase: alt.atheism
  Top 3 | Doc 9623 | Similaridad: 0.2288 | Clase: talk.politics.mideast
  Top 4 | Doc 8726 | Similaridad: 0.2243 | Clase: talk.politics.mideast
  Top 5 | Doc 11305 | Similaridad: 0.2239 | Clase: talk.politics.mideas

### DOCUMENTO 10568 — Clase: talk.religion.misc
La mayor similitud (cossim= 0.2893) se da con el documento 6177. Ambos poseen la misma clase, en el contenido ambos hablan sobre terrorismo y la relacion con la religion. 

### DOCUMENTO 1165 — Clase: comp.sys.mac.hardware
La mayor similitud (cossim= 0.2830) se da con el documento 2051, pertenece a la clase misc.forsale. Si bien las clases son distintas, ambos documentos hablan sobre un "VGA monitor". La similitud se da por el lenguaje aunque uno es una consulta tecnica y el otro una venta.

### DOCUMENTO 1506 — Clase: comp.os.ms-windows.misc
La mayor similitud (cossim= 0.9964) se da con el documento 144, que pertenece a la misma clase. Ambos documentos contienen datos binarios codificados (posiblemente partes de un archivo uuencodeado o base64), lo que explica la similitud extremadamente alta.

### DOCUMENTO 5139 — Clase: comp.sys.ibm.pc.hardware
La mayor similitud (cossim= 0.2451) se da con el documento 8928, que pertenece a la misma clase. El documento 5139 pregunta cómo conectar los cables del speaker de la PC a una placa SoundBlaster, mientras que el documento 8928 responde sobre la conexión de cables del speaker al reemplazar un motherboard.

### DOCUMENTO 8559 — Clase: comp.windows.x
La mayor similitud (cossim= 0.2040) se da con el documento 9000, que pertenece a la clase talk.politics.guns. Las clases son completamente distintas: el documento 8559 habla sobre archivos multilingües, mientras que el documento 9000 trata sobre política de armas y una carta al editor. En este caso no se encuentra similitud en los contenidos. 

# Punto 2


In [53]:
sim_matrix = cosine_similarity(X_test, X_train) 
nearest_train_idx = np.argmax(sim_matrix, axis=1)
y_pred_proto = y_train[nearest_train_idx]

f1_macro_proto = f1_score(y_test, y_pred_proto, average='macro')
acc_proto = np.mean(y_test == y_pred_proto)

print(f"F1 macro (prototipos): {f1_macro_proto:.4f}")
print(f"Accuracy (prototipos): {acc_proto:.4f}")


for i in range(5):
    j = nearest_train_idx[i]
    sim = sim_matrix[i, j]
    true_label = newsgroups_test.target_names[y_test[i]]
    pred_label = newsgroups_train.target_names[y_pred_proto[i]]
    print(f"Test {i}: true={true_label} | pred={pred_label} | sim={sim:.4f}")

F1 macro (prototipos): 0.5050
Accuracy (prototipos): 0.5089
Test 0: true=rec.autos | pred=alt.atheism | sim=0.2596
Test 1: true=comp.windows.x | pred=talk.religion.misc | sim=0.2342
Test 2: true=alt.atheism | pred=talk.politics.mideast | sim=0.3391
Test 3: true=talk.politics.mideast | pred=talk.politics.mideast | sim=0.4253
Test 4: true=talk.religion.misc | pred=alt.atheism | sim=0.2761


El clasificador por prototipos asigna a cada documento de test la clase del documento de entrenamiento más similar por coseno en el espacio TF-IDF.

- F1 macro: 0.5050
- Accuracy: 0.5089

Es un desempeño razonable para un metodo simple de vecino mas cercano. Tiene un costo computacional asociado a comprar el train contra el test. El el F1 macro: 0.5 nos dice que el rendimiento no es uniforme en todas las clases, algunas se separan bien y en otras se confunden con frecuencia.


### Punto 3

In [54]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test  = fetch_20newsgroups(subset='test',  remove=('headers', 'footers', 'quotes'))
y_train = newsgroups_train.target
y_test  = newsgroups_test.target

experiments = [
    ("Baseline: TF-IDF default + MultinomialNB",
     TfidfVectorizer(),
     MultinomialNB()),

    ("TF-IDF sublinear_tf + MultinomialNB",
     TfidfVectorizer(sublinear_tf=True),
     MultinomialNB()),

    ("TF-IDF sublinear_tf + ComplementNB",
     TfidfVectorizer(sublinear_tf=True),
     ComplementNB()),

    ("TF-IDF sublinear_tf + ComplementNB (norm=True)",
     TfidfVectorizer(sublinear_tf=True),
     ComplementNB(norm=True)),

    ("TF-IDF sublinear_tf + min_df=2 + max_df=0.8 + ComplementNB",
     TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.8),
     ComplementNB()),

    ("CountVectorizer + ComplementNB",
     CountVectorizer(),
     ComplementNB()),
]

results = []
for name, vect, clf in experiments:
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)
    clf.fit(Xtr, y_train)
    f1 = f1_score(y_test, clf.predict(Xte), average='macro')
    results.append((f1, name))
    print(f"F1 macro = {f1:.4f}  |  {name}")

best_f1, best_name = max(results)
print(f"\n>>> MEJOR: F1 macro = {best_f1:.4f}  |  {best_name}")

F1 macro = 0.5854  |  Baseline: TF-IDF default + MultinomialNB
F1 macro = 0.5860  |  TF-IDF sublinear_tf + MultinomialNB
F1 macro = 0.6920  |  TF-IDF sublinear_tf + ComplementNB
F1 macro = 0.6860  |  TF-IDF sublinear_tf + ComplementNB (norm=True)
F1 macro = 0.6934  |  TF-IDF sublinear_tf + min_df=2 + max_df=0.8 + ComplementNB
F1 macro = 0.6374  |  CountVectorizer + ComplementNB

>>> MEJOR: F1 macro = 0.6934  |  TF-IDF sublinear_tf + min_df=2 + max_df=0.8 + ComplementNB


In [55]:
# Ajuste fino del parametro alpha en la mejor configuracion
best_vect = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.8)
Xtr_best = best_vect.fit_transform(newsgroups_train.data)
Xte_best = best_vect.transform(newsgroups_test.data)

alphas = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 1.0, 2.0, 5.0]
alpha_results = []
for alpha in alphas:
    clf_a = ComplementNB(alpha=alpha)
    clf_a.fit(Xtr_best, y_train)
    f1 = f1_score(y_test, clf_a.predict(Xte_best), average='macro')
    alpha_results.append((f1, alpha))
    print(f"alpha={alpha:<5}  F1 macro = {f1:.4f}")

best_f1_a, best_alpha = max(alpha_results)
print(f"\n>>> Mejor alpha: {best_alpha}  →  F1 macro = {best_f1_a:.4f}")

alpha=0.01   F1 macro = 0.6743
alpha=0.05   F1 macro = 0.6853
alpha=0.1    F1 macro = 0.6899
alpha=0.3    F1 macro = 0.6957
alpha=0.5    F1 macro = 0.6963
alpha=0.7    F1 macro = 0.6944
alpha=1.0    F1 macro = 0.6934
alpha=2.0    F1 macro = 0.6873
alpha=5.0    F1 macro = 0.6752

>>> Mejor alpha: 0.5  →  F1 macro = 0.6963


Consideracion de los efectos de cada parametro del vectorizador.


Sublinear_tf=True : Aplica log(1+tf). Comprime el efecto de palabras muy frecuentes en un documento. Leve mejoria sobre MultinomialNB

min_df=2: Elimina terminos que aparecen en un solo documento. Reduce el vocabulario y el ruido mejorando la robustez

max_df=0.8: Ignora palabras que aparecen en mas del 80% de los documentos.

CountVectorizer: Sin normalizacion TF-IDF pierde potencia descriptiva.

ComplementNB supera ampliamente a MultinomialNB (~+10 puntos de F1 macro). ComplementNB modela la probabilidad de no pertenecer a cada clase, lo que lo hace más robusto ante clases con vocabulario desbalanceado.

El parámetro alpha controla el suavizado bayesiano. Valores muy pequeños (0.01) sobre-ajustan al vocabulario de entrenamiento. Valores grandes (5.0) nivelan demasiado las probabilidades. El óptimo encontrado es alpha = 0.5, que balancea ambos extremos.

Mejor configuración final: TF-IDF (sublinear_tf=True, min_df=2, max_df=0.8) + ComplementNB (alpha=0.5) → F1 macro = 0.6963 (mejora del +18.9 % sobre la baseline)


## Punto 4

In [56]:
X_term_doc = X_train.T  # shape: (vocab_size, n_documents)

print(f"Matriz término-documento: {X_term_doc.shape}")
print(f"  {X_term_doc.shape[0]} palabras  x  {X_term_doc.shape[1]} documentos\n")

words = ['bible', 'monitor', 'war', 'restaurant', 'basketball']

for word in words:
    if word not in tfidfvect.vocabulary_:
        print(f"'{word}' no está en el vocabulario — omitida")
        continue

    word_idx = tfidfvect.vocabulary_[word]
    word_vec = X_term_doc[word_idx]          # vector 1 x n_documents

    sim_all = cosine_similarity(word_vec, X_term_doc)[0]

    top5_idx = np.argsort(sim_all)[::-1][1:6]

    print(f"{'='*55}")
    print(f"  Palabra: '{word}'")
    print(f"  5 más similares:")
    for rank, idx in enumerate(top5_idx, 1):
        print(f"    {rank}. '{idx2word[idx]}'  (cos sim = {sim_all[idx]:.4f})")
print("=" * 55)

Matriz término-documento: (101631, 11314)
  101631 palabras  x  11314 documentos

  Palabra: 'bible'
  5 más similares:
    1. 'god'  (cos sim = 0.2616)
    2. 'christians'  (cos sim = 0.2473)
    3. 'contadictions'  (cos sim = 0.2189)
    4. 'necessitate'  (cos sim = 0.2087)
    5. 'c5elp2'  (cos sim = 0.1981)
  Palabra: 'monitor'
  5 más similares:
    1. 'vga'  (cos sim = 0.2846)
    2. 'monitors'  (cos sim = 0.2691)
    3. 'mag'  (cos sim = 0.2139)
    4. 'nanao'  (cos sim = 0.2061)
    5. '560i'  (cos sim = 0.2005)
  Palabra: 'war'
  5 más similares:
    1. 'irag'  (cos sim = 0.2548)
    2. 'dresden'  (cos sim = 0.2376)
    3. '1948'  (cos sim = 0.2366)
    4. 'lauches'  (cos sim = 0.2071)
    5. 'drugs'  (cos sim = 0.1981)
  Palabra: 'restaurant'
  5 más similares:
    1. 'fryer'  (cos sim = 0.5714)
    2. 'shailesh'  (cos sim = 0.5701)
    3. 'pans'  (cos sim = 0.5580)
    4. 'shelves'  (cos sim = 0.5109)
    5. 'puked'  (cos sim = 0.4410)
  Palabra: 'basketball'
  5 más similar


Al transponer la matriz documento-término obtenemos una matriz término-documento de shape (101 631 palabras × 11 314 documentos). Cada fila es el vector de una palabra: sus componentes indican en qué documentos (y con qué peso TF-IDF) aparece ese término. La similaridad coseno entre dos filas refleja cuánto co-ocurren esas palabras en los mismos documentos, funcionando como un embedding distribucional simple.


Para las 5 palabras seleccionadas, se encuentran términos que se asocian claramente. Bible con god y christians, mientras que c5elp2 tiene menos sentido pareciera ruido. Monitor se relaciona con todas, probablemente 560i hace referencia a un modelo especifico. Con war se pueden observar dos cosas, referencia hechos bélicos históricos como 1948 como también el concepto de lucha contra las drogas y por eso aparece drugs. Restaurant no tiene un newsgroup dedicado en el dataset y las palabras que encuentra tienen una clara relación con la cocina. Por último basketball encuentra claramente nba y equipos de la misma mientras que aparecen palabras que se relacionan poco como capitalize y 7224.